<img align="left" src="https://docs.linea.org.br/images/logo-linea.fw_2.png" width=100 style="padding: 40px">  
<img align="left" src="https://cdn2.webdamdb.com/1280_c3PXjCZbPM23.png" width=180>

# Photo-z final-product QA
<font size=4>Configuration-driven QA of HATS point estimates and companion or Nested PDFs</font>

This notebook performs configurable final-product QA over projected scalar and PDF columns. DP2-v3 scans the complete HATS catalog for scalar redshift distributions and science selections. The much larger Nested PDFs are inspected on a bounded technical subset, because every object carries two 601-value arrays. It checks product integrity and reconstructs HATS summaries from those matched PDF-QA partitions.

Throughout this report, **HATS** means the final `pz_summary` catalog. PDFs may come from a companion Parquet product or an algorithm-specific `*_ens_interp/{xvals,yvals}` Nested column in that same catalog. DP2-v3 opens the HATS collection root with `lsdb.open_catalog()`, which selects `pz_summary` and attaches its default margin without adding margin rows to the scan. A raw recursive Parquet reader must instead start at `pz_summary/dataset`; reading the collection root recursively would duplicate boundary objects. The paths report below identifies the active layout and scan boundary.

Paths, cuts, thresholds, and cluster resources come from YAML. This remains a compute and scientific-smoke QA: bias, NMAD, outlier rate, PIT, and coverage require truth redshifts and are intentionally outside its scope.

## 1. Select and inspect the configuration

Set `PZ_QA_CONFIG` before starting Jupyter or change the fallback below. Relative paths in YAML are resolved from the configuration file.

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import Markdown, display
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "pz_qa.py").exists():
    NOTEBOOK_DIR = Path("doc/notebooks").resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

from pz_qa import load_config, plot_nz, plot_zmean_selections, run_qa, start_cluster, write_report

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

CONFIG_PATH = Path(os.environ.get(
    "PZ_QA_CONFIG",
    NOTEBOOK_DIR / "config/dp1-v2-fzboost.yaml",
)).expanduser().resolve()
config = load_config(CONFIG_PATH)
display(Markdown("### Run metadata"))
display(pd.Series(config["run"], name="value").rename_axis("property").to_frame())
display(Markdown("### Science samples included in every result"))
display(pd.DataFrame(config["selections"]).set_index("name"))

## 2. Start Dask

The YAML supports `local`, `slurm`, and `existing`. Adaptive SLURM settings are kept with each run configuration. Cluster information shown below describes computational execution only; it is not a scientific result.

In [ ]:
client, cluster = start_cluster(config)
display(Markdown("**Dask cluster started successfully.**"))
display(pd.Series(config["cluster"], name="value").rename_axis("setting").to_frame())

## 3. Run the distributed QA

Each worker returns only fixed-size histograms, counters, moments, identifier digests, and a small deterministic cross-check sample. Enabled quality selections share one global scalar pass, avoiding a new catalog scan per cut. In the bounded Nested-PDF subset, scalar and PDF diagnostics share one task and one heavy partition read so their technical comparisons use identical rows. Only compact reductions return to the notebook process.

In [ ]:
result = run_qa(config)
display(Markdown("**Distributed QA completed.** The following sections contain only reduced metrics, checks, and plots; full catalogs and PDF matrices were not collected into the notebook process."))
scan_is_global = result.paths["HATS partitions inspected for science metrics"] == result.paths["HATS partitions"]
scan_description = "all HATS partitions (global science metrics)" if scan_is_global else "a bounded subset of HATS partitions"
pdf_qa_is_global = result.paths["HATS partitions inspected for PDF QA"] == result.paths["HATS partitions"]
pdf_qa_description = (
    "all HATS partitions" if pdf_qa_is_global
    else f"{result.paths['HATS partitions inspected for PDF QA']} of {result.paths['HATS partitions']} HATS partitions (bounded technical coverage)"
)
sample_labels = [item.get("label", item["name"]) for item in config["selections"]]
display(Markdown(
    "### Report at a glance\n"
    f"- **Product:** {config['run']['name']}\n"
    f"- **Catalog coverage:** {scan_description}\n"
    f"- **PDF source:** {result.paths['PDF layout']} layout — {result.paths['PDF source']}\n"
    f"- **PDF technical QA coverage:** {pdf_qa_description}\n"
    f"- **Science samples:** " + "; ".join(sample_labels)
))

## 4. Input products and integrity checks

The QA reads two distinct final products:

- **HATS `pz_summary`:** `objectId`, sky coordinates, `zmode`, `zmean`, `zmedian`, `z_p16`, `z_p84`, and columns used by quality selections. DP2-v3 includes `refExtendedness`, where `0` is point-like, `1` is extended, and null is unclassified. HATS supplies the point-estimate statistics, interval diagnostics, coordinates, and point-estimate N(z).
- **PDF source:** in companion mode, Parquet shards provide `objectId`, fixed-size `pdf` vectors, selection columns, and an `xvals.parquet` grid. In DP2-v3 Nested mode, the algorithm-specific `*_ens_interp` column in HATS provides per-object `xvals` and `yvals`. The PDF coverage is reported explicitly above; DP2-v3 limits these large vectors to bounded technical integrity diagnostics and reconstructed summaries. PDFs are not needed for the global scalar N(z) plots.

Checks mentioning both products are cross-product checks. The `objectId` comparison uses count plus three commutative 64-bit digests, avoiding a catalog-wide shuffle unless exact uniqueness is enabled in YAML.

In [ ]:
display(Markdown("### Configured inputs"))
paths_table = result.paths.rename_axis("item").reset_index(name="value")
display(paths_table.style.set_properties(
    subset=["value"],
    **{"white-space": "normal", "overflow-wrap": "anywhere", "text-align": "left"},
))

display(Markdown("### Redshift grid — from the configured PDF source"))
display(result.grid.rename_axis("property").to_frame())

display(Markdown("### Integrity checks — HATS scalars, configured PDF source, and cross-product"))
display(result.checks.style.map(
    lambda value: "background-color: #d8f3dc" if value else "background-color: #ffccd5",
    subset=["passed"],
))

## 5. Global science metrics and technical PDF diagnostics

The tables below deliberately separate scalar science results from PDF product diagnostics. HATS point-estimate metrics and interval-validity counters use the complete catalog; the interval-width median is explicitly a deterministic bounded sample statistic. PDF metrics, raw sums, and reconstructed summaries use the PDF-QA coverage reported above: all companion shards in companion mode, or the first configured number of HATS partitions in DP2-v3 Nested mode. That deterministic spatial subset is chosen for reproducible product checks, not as a statistically representative sample. PDF-derived quantities are therefore not presented as global scientific redshift distributions. PDF-sum quantiles also use a deterministic bounded sample.

For the global scalar distributions, `count`, `mean`, and `std` use every finite value, including any value outside the configured PDF grid. The reported `p16`, `median`, and `p84` are grid-binned percentiles of the finite values inside that grid; `outside_grid_fraction` discloses any excluded tail. Scalar `low_z_fraction` is counted directly from all finite values at or below the configured threshold, rather than inferred from histogram bins. For stacked PDFs, the analogous fraction is posterior mass on grid points at or below that threshold. `lower_edge_fraction` and `upper_edge_fraction` isolate the first and last grid bins, while `edge_fraction` combines the configured number of bins at both boundaries. Invalid-vector counters cover every PDF-QA row; raw-sum statistics and the stack include only valid PDFs, each normalized to unit sum before stacking. The PDF metric `count` reports that valid population.

The cross-product shape table uses matched PDF-QA rows on both sides: point estimates from the reported PDF coverage versus their stacked PDFs. A distance is emitted only when the selected `objectId` fingerprints match, every selected PDF is valid, and every corresponding scalar estimate is finite and inside the common grid, so the two curves represent exactly the same objects. Total variation is dimensionless and ranges from 0 for identical normalized histograms to 1 for disjoint ones; `wasserstein_z` is a redshift-scale displacement. These are technical consistency diagnostics, not global N(z) measurements or accuracy metrics against truth.

In [ ]:
hats_metrics = result.global_metrics
pdf_metrics = result.pdf_metrics

hats_health_columns = [
    "HATS rows", "interval width median (sample)",
    "invalid coordinates", "invalid intervals",
    "non-finite intervals", "intervals outside grid",
]
pdf_health_columns = [
    "PDF QA HATS rows", "PDF rows", "valid PDF fraction", "wrong length", "non-finite",
    "negative", "non-positive sum", "PDF sum mean", "PDF sum std",
    "PDF sum min", "PDF sum max", "PDF sum p01 (sample)",
    "PDF sum median (sample)", "PDF sum p99 (sample)",
]
if result.paths["PDF layout"] == "nested":
    pdf_health_columns[6:6] = ["invalid nested xvals", "non-float32 nested yvals"]

low_z_threshold = config["metrics"].get("low_z_threshold", 0.1)
display(Markdown(f"**Low-redshift diagnostic:** scalar `low_z_fraction` is the exact fraction of finite estimates with $z \\leq {low_z_threshold:g}$ over the complete HATS catalog. The stacked-PDF value is posterior mass on grid points satisfying the same condition and uses {pdf_qa_description}."))

display(Markdown("### From the HATS `pz_summary` catalog"))
display(hats_metrics)
display(result.pdf_health[hats_health_columns])

display(Markdown(f"### PDF product diagnostics (`{result.paths['PDF layout']}` layout; {pdf_qa_description})"))
display(pdf_metrics)
display(result.pdf_health[pdf_health_columns])

display(Markdown("### Matched-coverage point-estimate versus PDF-stack comparison"))
display(result.distribution_distances)

### Cross-product check: HATS summaries reconstructed from the configured PDFs

Within the reported PDF-QA coverage, a deterministic sample keyed by `objectId` recomputes `zmode`, `zmean`, `zmedian`, `z_p16`, and `z_p84` from valid configured PDF vectors and compares them with the stored HATS columns for the same objects. In DP2-v3 these are the Nested `yvals`; in companion mode they are the Parquet `pdf` vectors. The values should agree up to the configured numerical tolerance because Stage 3 derives the HATS summaries from the same PDFs. This is a product-consistency test, not a global N(z) or an accuracy measurement against truth.

In [ ]:
display(Markdown("`compared` is the number of matched sampled objects. Absolute differences are in redshift units; `fraction_over_tolerance` uses the tolerance configured in YAML."))
display(result.crosscheck)

## 6. Global N(z)

Both panels scan the complete HATS catalog. Histogram bins contain finite point estimates inside the configured redshift range; non-finite and outside-grid fractions are reported separately in the global metrics table. The left panel shows counts, while the right panel compares shapes normalized over that common displayed support for `zmode`, `zmean`, and `zmedian` in every enabled science selection. Nested PDF vectors are not read for these plots.

For readability, the standard configuration uses a robust vertical display range based on the 99.5th percentile of positive bin heights. Values are **not removed, clipped in the calculations, or excluded from any metric**. Whenever a peak exceeds the displayed range, its estimator, amplitude, and redshift are printed inside the panel. Set `plots.nz_ymax_quantile: 1.0` in YAML to restore the full linear range.

In [ ]:
display_quantile = config.get("plots", {}).get("nz_ymax_quantile", 0.995)
display(Markdown(f"**Plot display:** robust y-range quantile = `{display_quantile:g}`; all off-scale maxima are annotated."))
figure = plot_nz(result)
figure

## 7. `zmean` distributions by science sample

For DP2-v3, the four panels answer a simple sequence of questions:

1. What is the `zmean` distribution with no cut?
2. What remains after requiring the stored SFD-corrected GAaP $i$-band magnitude `i_gaap1p0Mag_sfd < 26`?
3. What remains after selecting extended sources with `refExtendedness == 1`?
4. What remains when both cuts are required?

`refExtendedness` is a reference-band morphological classifier: `0` denotes point-like, `1` denotes extended, and null values are unclassified. The extended sample is therefore a practical **galaxy proxy**, not a truth-level galaxy sample; no claim of perfect stellar rejection, completeness, or purity is made. Objects with null classification are excluded from both extended-source panels.

The fourth sample is the logical intersection of the second and third. Non-finite magnitudes are excluded from the two magnitude-limited samples. The retention table counts selected catalog rows; each plot legend reports the number with finite `zmean` inside the configured redshift range. Vertical scales are independent so smaller samples remain readable; use the retention table when comparing absolute populations. Display-only y-axis limits may hide the top of an isolated peak, but every such peak is annotated and all bins remain in the calculations. These scientific distributions always use every HATS partition.

In [ ]:
all_selection = next(item for item in config["selections"] if not item.get("filters"))
all_label = all_selection.get("label", all_selection["name"])
sample_composition = result.pdf_health[["HATS rows"]].rename(columns={"HATS rows": "objects"})
sample_composition["fraction of all objects"] = (
    sample_composition["objects"] / sample_composition.loc[all_label, "objects"]
)
display(Markdown("### Sample sizes and retention"))
display(sample_composition.style.format({"objects": "{:,.0f}", "fraction of all objects": "{:.2%}"}))
display(Markdown("### Global `zmean` count distributions"))
zmean_figure = plot_zmean_selections(result)
display(zmean_figure)

## 8. Save compact reports and release resources

Set `paths.report_dir` in YAML to write CSV/JSON summaries. Full catalogs and PDF matrices are never written or collected here.

In [ ]:
report_dir = write_report(result)
print(f"Report directory: {report_dir}" if report_dir else "Report writing disabled.")
print(f"Checks passed: {int(result.checks['passed'].sum())}/{len(result.checks)}")

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print("Dask resources released.")

### Acknowledgments

*This notebook was supported by resources supplied by the Laboratório Interinstitucional de e-Astronomia (LIneA).*  
*These results were obtained using RAIL, an open-source software package made available by the LSST Dark Energy Science Collaboration (DESC).*